In [1]:
import pandas as pd

dribbles_clean = pd.read_csv('data/processed/wc2022sb_dribbles_clean.csv', low_memory=False)
print(dribbles_clean.shape)
print(dribbles_clean.columns.tolist())

(1793, 17)
['id', 'match_id', 'player', 'player_id', 'team', 'team_id', 'minute', 'second', 'period', 'x', 'y', 'dribble_nutmeg', 'dribble_outcome', 'dribble_overrun', 'dribble_no_touch', 'play_pattern', 'position']


In [2]:
player_profile = dribbles_clean.groupby('player_id').agg(
    total_dribbles=('dribble_outcome', 'count'),
    successful_dribbles=('dribble_outcome', lambda x: (x == 'Complete').sum()),
    total_nutmegs=('dribble_nutmeg', 'sum'),
    total_overrun=('dribble_overrun', 'sum'),
).reset_index()

player_profile['dribble_success_rate'] = (
    player_profile['successful_dribbles'] / player_profile['total_dribbles']
)
player_profile['nutmeg_rate'] = (
    player_profile['total_nutmegs'] / player_profile['total_dribbles']
)

print(player_profile.shape)
print(player_profile.sort_values('total_nutmegs', ascending=False).head(10))

(438, 7)
     player_id  total_dribbles  successful_dribbles  total_nutmegs  \
6       3009.0              50                   30              5   
312    25363.0              15                    5              4   
309    25104.0               8                    2              3   
261    16532.0              18                    4              3   
72      4320.0              21                   10              3   
276    20750.0              13                    5              3   
117     5474.0              17                    8              3   
5       2995.0              29                   18              3   
302    24024.0              23                   10              2   
303    24085.0               8                    3              2   

     total_overrun  dribble_success_rate  nutmeg_rate  
6                4              0.600000     0.100000  
312              1              0.333333     0.266667  
309              2              0.250000     0.37500

In [3]:
print(player_profile['total_dribbles'].describe())
print(f"\nPemain dengan < 5 dribbles: {(player_profile['total_dribbles'] < 5).sum()}")
print(f"Pemain dengan >= 5 dribbles: {(player_profile['total_dribbles'] >= 5).sum()}")

count    438.000000
mean       4.093607
std        5.074088
min        1.000000
25%        1.000000
50%        2.000000
75%        5.000000
max       50.000000
Name: total_dribbles, dtype: float64

Pemain dengan < 5 dribbles: 320
Pemain dengan >= 5 dribbles: 118


Prediksi berapa kali pemain dengan gaya bermain flair — winger, striker, CAM akan melakukan nutmeg dalam satu pertandingan

In [4]:
# Ambil semua pemain yang ada di WC 2022
wc_players = set(dribbles_clean['player_id'].dropna().unique())
print(f"Total pemain unik di WC 2022 dribbles: {len(wc_players)}")

Total pemain unik di WC 2022 dribbles: 438


In [5]:
print(dribbles_clean['position'].value_counts())

position
Left Wing                    256
Right Wing                   242
Center Forward               127
Center Attacking Midfield    110
Left Back                    107
Right Midfield                94
Left Center Midfield          87
Right Center Midfield         84
Right Defensive Midfield      83
Right Back                    83
Right Center Forward          78
Left Midfield                 71
Left Defensive Midfield       66
Left Center Forward           65
Left Wing Back                52
Right Wing Back               51
Center Defensive Midfield     35
Right Center Back             32
Left Center Back              29
Left Attacking Midfield       17
Right Attacking Midfield      16
Center Back                    5
Goalkeeper                     3
Name: count, dtype: int64


In [6]:
flair_positions = [
    'Left Wing', 'Right Wing',
    'Center Forward',
    'Center Attacking Midfield',
    'Left Center Forward', 'Right Center Forward',
    'Left Attacking Midfield', 'Right Attacking Midfield',
    'Left Midfield', 'Right Midfield'
]

flair_dribbles = dribbles_clean[dribbles_clean['position'].isin(flair_positions)].copy()

print(f"Total dribbles (semua posisi): {len(dribbles_clean)}")
print(f"Total dribbles (flair positions): {len(flair_dribbles)}")
print(f"Total nutmegs (flair positions): {flair_dribbles['dribble_nutmeg'].sum()}")
print(f"Nutmeg rate (flair positions): {flair_dribbles['dribble_nutmeg'].mean()*100:.1f}%")

Total dribbles (semua posisi): 1793
Total dribbles (flair positions): 1076
Total nutmegs (flair positions): 89
Nutmeg rate (flair positions): 8.3%


In [7]:
flair_profile = flair_dribbles.groupby(['player_id', 'player', 'team']).agg(
    total_dribbles=('dribble_outcome', 'count'),
    successful_dribbles=('dribble_outcome', lambda x: (x == 'Complete').sum()),
    total_nutmegs=('dribble_nutmeg', 'sum'),
    total_overrun=('dribble_overrun', 'sum'),
).reset_index()

flair_profile['dribble_success_rate'] = (
    flair_profile['successful_dribbles'] / flair_profile['total_dribbles']
)
flair_profile['nutmeg_rate'] = (
    flair_profile['total_nutmegs'] / flair_profile['total_dribbles']
)

# Filter minimum 5 dribble attempts
flair_profile_filtered = flair_profile[flair_profile['total_dribbles'] >= 5].copy()

print(f"Total flair players: {len(flair_profile)}")
print(f"Flair players dengan >= 5 dribbles: {len(flair_profile_filtered)}")
print(flair_profile_filtered.sort_values('total_nutmegs', ascending=False).head(10))

Total flair players: 215
Flair players dengan >= 5 dribbles: 71
     player_id                           player         team  total_dribbles  \
4       3009.0             Kylian Mbappé Lottin       France              50   
157    25363.0        Antony Matheus dos Santos       Brazil              15   
3       2995.0  Ángel Fabián Di María Hernández    Argentina              29   
155    25104.0            Rodrygo Silva de Goes       Brazil               8   
134    20750.0                Cody Mathès Gakpo  Netherlands              13   
37      4320.0    Neymar da Silva Santos Junior       Brazil              21   
0       2941.0                     Ismaïla Sarr      Senegal              13   
185    39565.0                    Jamal Musiala      Germany              21   
114    12365.0                  Alphonso Davies       Canada              19   
125    16532.0             Daniel Olmo Carvajal        Spain              16   

     successful_dribbles  total_nutmegs  total_overrun 

In [8]:
flair_profile_filtered.to_csv('data/processed/flair_player_profiles.csv', index=False)
print("Tersimpan!")

Tersimpan!


In [9]:
import pandas as pd

all_events = pd.read_csv('data/raw/all_events_combined.csv', low_memory=False)
print(all_events.shape)

(522885, 116)


In [10]:
# Filter dribbles dari semua kompetisi
all_dribbles = all_events[all_events['type'] == 'Dribble'].copy()

# Fix missing values
all_dribbles['dribble_nutmeg'] = all_dribbles['dribble_nutmeg'].fillna(False)
all_dribbles['dribble_overrun'] = all_dribbles['dribble_overrun'].fillna(False)
all_dribbles['dribble_no_touch'] = all_dribbles['dribble_no_touch'].fillna(False)

print(f"Total dribbles: {len(all_dribbles)}")
print(f"Total nutmegs: {all_dribbles['dribble_nutmeg'].sum()}")
print(f"Nutmeg rate: {all_dribbles['dribble_nutmeg'].mean()*100:.1f}%")
print(f"\nPer kompetisi:")
print(all_dribbles.groupby('competition')['dribble_nutmeg'].agg(['count','sum','mean']).round(3))

Total dribbles: 4027
Total nutmegs: 354
Nutmeg rate: 8.8%

Per kompetisi:
                   count  sum      mean
competition                            
Copa America 2024    945   79  0.083598
Euro 2024           1289  130  0.100853
WC 2022             1793  145   0.08087


In [11]:
flair_positions = [
    'Left Wing', 'Right Wing',
    'Center Forward',
    'Center Attacking Midfield',
    'Left Center Forward', 'Right Center Forward',
    'Left Attacking Midfield', 'Right Attacking Midfield',
    'Left Midfield', 'Right Midfield'
]

flair_dribbles_all = all_dribbles[all_dribbles['position'].isin(flair_positions)].copy()

flair_profile_all = flair_dribbles_all.groupby(['player_id', 'player', 'team']).agg(
    total_dribbles=('dribble_outcome', 'count'),
    successful_dribbles=('dribble_outcome', lambda x: (x == 'Complete').sum()),
    total_nutmegs=('dribble_nutmeg', 'sum'),
    total_overrun=('dribble_overrun', 'sum'),
).reset_index()

flair_profile_all['dribble_success_rate'] = (
    flair_profile_all['successful_dribbles'] / flair_profile_all['total_dribbles']
)
flair_profile_all['nutmeg_rate'] = (
    flair_profile_all['total_nutmegs'] / flair_profile_all['total_dribbles']
)

flair_profile_filtered_all = flair_profile_all[flair_profile_all['total_dribbles'] >= 5].copy()

print(f"Total flair players: {len(flair_profile_all)}")
print(f"Flair players dengan >= 5 dribbles: {len(flair_profile_filtered_all)}")
print(flair_profile_filtered_all.sort_values('total_nutmegs', ascending=False).head(10))

Total flair players: 404
Flair players dengan >= 5 dribbles: 153
     player_id                           player         team  total_dribbles  \
238    25104.0            Rodrygo Silva de Goes       Brazil              23   
3       2995.0  Ángel Fabián Di María Hernández    Argentina              47   
359    68574.0        Nicholas Williams Arthuer        Spain              45   
396   316046.0      Lamine Yamal Nasraoui Ebana        Spain              27   
4       3009.0             Kylian Mbappé Lottin       France              77   
320    39565.0                    Jamal Musiala      Germany              42   
188    16528.0                 Petar Stojanović     Slovenia              11   
189    16532.0             Daniel Olmo Carvajal        Spain              29   
201    20750.0                Cody Mathès Gakpo  Netherlands              24   
110     7156.0                  Federico Chiesa        Italy              13   

     successful_dribbles total_nutmegs total_overrun  

In [12]:
flair_profile_filtered_all.to_csv('data/processed/flair_player_profiles_all.csv', index=False)
print("Tersimpan!")

Tersimpan!


In [13]:
import pandas as pd

all_events = pd.read_csv('data/raw/all_events_combined.csv', low_memory=False)

flair_positions = [
    'Left Wing', 'Right Wing', 'Center Forward',
    'Center Attacking Midfield', 'Left Center Forward',
    'Right Center Forward', 'Left Attacking Midfield',
    'Right Attacking Midfield', 'Left Midfield', 'Right Midfield'
]

all_dribbles = all_events[all_events['type'] == 'Dribble'].copy()
all_dribbles['dribble_nutmeg'] = all_dribbles['dribble_nutmeg'].fillna(False)
all_dribbles['dribble_overrun'] = all_dribbles['dribble_overrun'].fillna(False)
all_dribbles['dribble_no_touch'] = all_dribbles['dribble_no_touch'].fillna(False)

flair_dribbles = all_dribbles[all_dribbles['position'].isin(flair_positions)].copy()
print(f"Ready: {len(flair_dribbles)} flair dribbles, {flair_dribbles['dribble_nutmeg'].sum()} nutmegs")

Ready: 2390 flair dribbles, 217 nutmegs


In [14]:
# Hitung berapa match yang dimainkan tiap pemain
matches_played = flair_dribbles.groupby('player_id')['match_id'].nunique().reset_index()
matches_played.columns = ['player_id', 'matches_played']

# Hitung total dribble attempts per pemain
total_attempts = flair_dribbles.groupby('player_id')['dribble_outcome'].count().reset_index()
total_attempts.columns = ['player_id', 'total_dribbles']

# Gabungkan dan hitung rata-rata per match
dribble_exposure = matches_played.merge(total_attempts, on='player_id')
dribble_exposure['dribbles_per_match'] = (
    dribble_exposure['total_dribbles'] / dribble_exposure['matches_played']
)

print(dribble_exposure.sort_values('dribbles_per_match', ascending=False).head(10))

     player_id  matches_played  total_dribbles  dribbles_per_match
46      4320.0               3              21            7.000000
309    38229.0               4              28            7.000000
4       3009.0              12              77            6.416667
225    23650.0               5              31            6.200000
338    47112.0               3              18            6.000000
80      5628.0               1               6            6.000000
194    18395.0               7              38            5.428571
335    44455.0               3              16            5.333333
319    39565.0               8              42            5.250000
162    12306.0               1               5            5.000000


In [15]:
# Gabungkan dengan info tanggal match
match_dates = all_events[['match_id', 'competition']].drop_duplicates()

flair_dribbles_dated = flair_dribbles.merge(match_dates, on='match_id', how='left')

# Hitung nutmeg per pemain per match
nutmeg_per_match = flair_dribbles_dated.groupby(['player_id', 'match_id']).agg(
    nutmegs=('dribble_nutmeg', 'sum'),
    attempts=('dribble_outcome', 'count')
).reset_index()

# Urutkan berdasarkan match_id (proxy urutan pertandingan)
nutmeg_per_match = nutmeg_per_match.sort_values(['player_id', 'match_id'])

# Hitung rolling rate 3 match terakhir
nutmeg_per_match['rolling_nutmeg_rate'] = (
    nutmeg_per_match.groupby('player_id')['nutmegs']
    .transform(lambda x: x.rolling(3, min_periods=1).mean())
)

# Ambil nilai terakhir per pemain (form terkini)
latest_form = nutmeg_per_match.groupby('player_id').last().reset_index()
latest_form = latest_form[['player_id', 'rolling_nutmeg_rate']]

print(latest_form.sort_values('rolling_nutmeg_rate', ascending=False).head(10))

     player_id  rolling_nutmeg_rate
187    16528.0             1.333333
285    33234.0             1.333333
358    68574.0             1.333333
109     7156.0             1.333333
237    25104.0             1.333333
82      5632.0             1.000000
79      5623.0             1.000000
3       2995.0             1.000000
307    37122.0             1.000000
30      3457.0             1.000000


ada masalah di sini. Nilai rolling_nutmeg_rate di atas 1.0 tidak masuk akal untuk sebuah rate — maksimalnya harusnya 1.0 (100%).
Ini terjadi karena menghitung rolling average of nutmeg counts, bukan rolling average of nutmeg rates. Jadi kalau pemain nutmeg 2 kali di satu pertandingan, nilainya bisa di atas 1.
Perbaiki dengan menghitung rate per match dulu, baru di-rolling:

In [16]:
# Hitung rate per match dulu
nutmeg_per_match['nutmeg_rate_match'] = (
    nutmeg_per_match['nutmegs'] / nutmeg_per_match['attempts']
)

# Rolling rate dari rate per match
nutmeg_per_match['rolling_nutmeg_rate'] = (
    nutmeg_per_match.groupby('player_id')['nutmeg_rate_match']
    .transform(lambda x: x.rolling(3, min_periods=1).mean())
)

# Ambil nilai terakhir per pemain
latest_form = nutmeg_per_match.groupby('player_id').last().reset_index()
latest_form = latest_form[['player_id', 'rolling_nutmeg_rate']]

print(latest_form.sort_values('rolling_nutmeg_rate', ascending=False).head(10))

     player_id  rolling_nutmeg_rate
23      3347.0                  1.0
340    49007.0                  1.0
30      3457.0                  1.0
55      5218.0                  1.0
207    21261.0                  1.0
175    15896.0                  1.0
82      5632.0                  1.0
401   405926.0                  1.0
98      6319.0                  1.0
208    21582.0                  1.0


Masih ada masalah — terlalu banyak pemain dengan nilai 1.0. Ini terjadi karena pemain yang hanya punya 1 pertandingan dengan 1 nutmeg dari 1 attempt langsung dapat rate 1.0.
Ini kembali ke masalah small sample yang kita diskusikan sebelumnya. Rolling rate tidak reliable untuk pemain dengan sedikit pertandingan.
Cek dulu distribusinya:

In [17]:
print(nutmeg_per_match.groupby('player_id')['match_id'].count().describe())
print(f"\nPemain dengan hanya 1 pertandingan: {(nutmeg_per_match.groupby('player_id')['match_id'].count() == 1).sum()}")
print(f"Pemain dengan >= 3 pertandingan: {(nutmeg_per_match.groupby('player_id')['match_id'].count() >= 3).sum()}")

count    403.000000
mean       2.526055
std        2.031148
min        1.000000
25%        1.000000
50%        2.000000
75%        3.000000
max       12.000000
Name: match_id, dtype: float64

Pemain dengan hanya 1 pertandingan: 171
Pemain dengan >= 3 pertandingan: 144


Ini konfirmasi masalahnya — 171 pemain hanya punya 1 pertandingan, dan median hanya 2 pertandingan. Rolling 3 match tidak bisa reliable di dataset setournament ini.
Tapi ini bukan berarti fitur rolling form harus dibuang. Kita perlu strategi fallback:

In [18]:
# Hitung jumlah match per pemain
match_counts = nutmeg_per_match.groupby('player_id')['match_id'].count().reset_index()
match_counts.columns = ['player_id', 'total_matches']

latest_form = nutmeg_per_match.groupby('player_id').last().reset_index()
latest_form = latest_form.merge(match_counts, on='player_id')

# Strategi fallback:
# >= 3 matches → pakai rolling_nutmeg_rate (reliable)
# < 3 matches → pakai overall nutmeg_rate dari flair_profile
flair_profile = pd.read_csv('data/processed/flair_player_profiles_all.csv')

latest_form = latest_form.merge(
    flair_profile[['player_id', 'nutmeg_rate']], 
    on='player_id', 
    how='left'
)

latest_form['form_nutmeg_rate'] = latest_form.apply(
    lambda x: x['rolling_nutmeg_rate'] if x['total_matches'] >= 3 else x['nutmeg_rate'],
    axis=1
)

print(latest_form[['player_id', 'total_matches', 'rolling_nutmeg_rate', 'nutmeg_rate', 'form_nutmeg_rate']]
      .sort_values('form_nutmeg_rate', ascending=False).head(10))

     player_id  total_matches  rolling_nutmeg_rate  nutmeg_rate  \
395   316046.0              7             0.500000     0.222222   
237    25104.0              8             0.455556     0.304348   
54      5207.0              7             0.444444     0.181818   
204    21209.0              3             0.400000     0.333333   
155    11174.0              3             0.400000     0.285714   
0       2941.0              4             0.388889     0.153846   
187    16528.0              4             0.388889     0.363636   
358    68574.0             10             0.355556     0.155556   
109     7156.0              3             0.344444     0.307692   
201    20758.0              3             0.333333     0.200000   

     form_nutmeg_rate  
395          0.500000  
237          0.455556  
54           0.444444  
204          0.400000  
155          0.400000  
0            0.388889  
187          0.388889  
358          0.355556  
109          0.344444  
201          0.333333 

Jauh lebih baik! Sekarang nilai-nilainya masuk akal dan semua di bawah 1.0.
Perhatikan player_id 316046 di puncak dengan form_nutmeg_rate 0.5 — itu Lamine Yamal. Pemain muda yang sedang dalam form luar biasa di Euro 2024.
Sekarang kita bangun fitur ketiga — Defensive Strength lawan. Ini mengukur seberapa sulit lawan untuk di-nutmeg:

In [19]:
# Hitung defensive strength per tim dari all_events
# Proxy: duel win rate dan pressure intensity

defensive_stats = all_events[all_events['type'].isin(['Duel', 'Pressure'])].copy()

# Duel win rate per tim
duel_stats = all_events[all_events['type'] == 'Duel'].copy()
duel_per_team = duel_stats.groupby('team').agg(
    total_duels=('type', 'count'),
    duels_won=('duel_outcome', lambda x: x.isin(['Won', 'Success']).sum())
).reset_index()
duel_per_team['duel_win_rate'] = duel_per_team['duels_won'] / duel_per_team['total_duels']

# Pressure intensity per tim per match
pressure_per_match = all_events[all_events['type'] == 'Pressure'].groupby(
    ['team', 'match_id']
).size().reset_index(name='pressure_count')
pressure_intensity = pressure_per_match.groupby('team')['pressure_count'].mean().reset_index()
pressure_intensity.columns = ['team', 'avg_pressure_per_match']

# Gabungkan
defensive_profile = duel_per_team.merge(pressure_intensity, on='team', how='left')

print(defensive_profile.sort_values('duel_win_rate', ascending=False).head(10))

           team  total_duels  duels_won  duel_win_rate  avg_pressure_per_match
50    Venezuela          153         28       0.183007              131.000000
6        Brazil          305         51       0.167213              127.888889
0       Albania           79         13       0.164557              146.666667
31         Peru          116         19       0.163793              158.666667
10     Colombia          216         34       0.157407              138.000000
18      Georgia          150         23       0.153333              128.500000
28  Netherlands          325         49       0.150769              125.090909
33     Portugal          290         42       0.144828              117.600000
9         Chile          111         16       0.144144              154.333333
17       France          385         55       0.142857              131.846154


ada yang aneh di sini. duel_win_rate tertinggi hanya 18% untuk Venezuela? Ini tidak masuk akal untuk sebuah "win rate".
Masalahnya ada di cara StatsBomb melabeli duel outcome? Mari kita cek dulu nilai unik yang ada:

In [20]:
print(all_events[all_events['type'] == 'Duel']['duel_outcome'].value_counts())

duel_outcome
Success In Play    1646
Lost In Play       1138
Won                1129
Lost Out            873
Success Out         168
Name: count, dtype: int64


Ketemu masalahnya! StatsBomb punya 5 kategori duel outcome, dan "menang" bukan hanya Won tapi juga Success In Play dan Success Out.
Perbaiki filternya:

In [21]:
winning_outcomes = ['Won', 'Success In Play', 'Success Out']

duel_per_team = all_events[all_events['type'] == 'Duel'].groupby('team').agg(
    total_duels=('type', 'count'),
    duels_won=('duel_outcome', lambda x: x.isin(winning_outcomes).sum())
).reset_index()

duel_per_team['duel_win_rate'] = duel_per_team['duels_won'] / duel_per_team['total_duels']

defensive_profile = duel_per_team.merge(pressure_intensity, on='team', how='left')

print(defensive_profile.sort_values('duel_win_rate', ascending=False).head(10))

         team  total_duels  duels_won  duel_win_rate  avg_pressure_per_match
50  Venezuela          153         62       0.405229              131.000000
6      Brazil          305        120       0.393443              127.888889
23      Italy           90         34       0.377778              147.000000
10   Colombia          216         79       0.365741              138.000000
49    Uruguay          321        116       0.361371              150.111111
17     France          385        137       0.355844              131.846154
1   Argentina          417        148       0.354916              137.000000
43      Spain          302        106       0.350993              142.818182
47    Ukraine           86         30       0.348837              139.666667
33   Portugal          290        101       0.348276              117.600000


Jauh lebih masuk akal sekarang! Brazil, Argentina, France, Spain, Portugal — tim-tim defensif kuat memang ada di atas.
Simpan defensive profile ini dan gabungkan semua fitur yang sudah kita bangun:

In [22]:
# Gabungkan semua fitur Level 1 + Level 2
flair_profile = pd.read_csv('data/processed/flair_player_profiles_all.csv')

# Gabung dengan dribble exposure
master_features = flair_profile.merge(dribble_exposure, on='player_id', how='left')

# Gabung dengan rolling form
master_features = master_features.merge(
    latest_form[['player_id', 'total_matches', 'form_nutmeg_rate']], 
    on='player_id', 
    how='left'
)

print(master_features.shape)
print(master_features.columns.tolist())

(153, 14)
['player_id', 'player', 'team', 'total_dribbles_x', 'successful_dribbles', 'total_nutmegs', 'total_overrun', 'dribble_success_rate', 'nutmeg_rate', 'matches_played', 'total_dribbles_y', 'dribbles_per_match', 'total_matches', 'form_nutmeg_rate']


153 pemain dengan 14 fitur — bagus! Tapi ada kolom duplikat: total_dribbles_x dan total_dribbles_y karena merge dua sumber yang sama-sama punya kolom total_dribbles.
Bersihkan dulu:

In [23]:
# Drop kolom duplikat dan rename
master_features = master_features.drop(columns=['total_dribbles_y'])
master_features = master_features.rename(columns={'total_dribbles_x': 'total_dribbles'})

print(master_features.columns.tolist())
print(master_features.head(3))

['player_id', 'player', 'team', 'total_dribbles', 'successful_dribbles', 'total_nutmegs', 'total_overrun', 'dribble_success_rate', 'nutmeg_rate', 'matches_played', 'dribbles_per_match', 'total_matches', 'form_nutmeg_rate']
   player_id         player         team  total_dribbles  successful_dribbles  \
0     2941.0   Ismaïla Sarr      Senegal              13                    9   
1     2972.0  Marcus Thuram       France              10                    5   
2     2988.0  Memphis Depay  Netherlands              24                    8   

   total_nutmegs  total_overrun  dribble_success_rate  nutmeg_rate  \
0              2              0              0.692308     0.153846   
1              0              2              0.500000     0.000000   
2              2              3              0.333333     0.083333   

   matches_played  dribbles_per_match  total_matches  form_nutmeg_rate  
0               4            3.250000              4          0.388889  
1               5        

Bersih! 13 kolom yang rapi dan bermakna.Sekarang tambahkan satu fitur terakhir Level 2 — fase turnamen. Ini proxy untuk tekanan pertandingan:

In [ ]:
# Ambil info competition stage dari matches
all_matches = all_events[['match_id', 'competition']].drop_duplicates()

# Hitung rata-rata nutmeg per pemain per fase
# Proxy sederhana: gunakan competition sebagai konteks
stage_features = flair_dribbles.merge(all_matches, on='match_id', how='left')
stage_features = stage_features.groupby(['player_id', 'competition']).agg(
    nutmegs_in_comp=('dribble_nutmeg', 'sum'),
    attempts_in_comp=('dribble_outcome', 'count')
).reset_index()

# Pivot: satu kolom per kompetisi
stage_pivot = stage_features.pivot_table(
    index='player_id',
    columns='competition',
    values='nutmegs_in_comp',
    fill_value=0
).reset_index()

stage_pivot.columns.name = None
stage_pivot = stage_pivot.rename(columns={
    'WC 2022': 'nutmegs_wc2022',
    'Euro 2024': 'nutmegs_euro2024',
    'Copa America 2024': 'nutmegs_copa2024'
})

print(stage_pivot.head())
print(stage_pivot.shape)

HASILNYA ERROR: Masalahnya adalah kolom competition tidak ada di flair_dribbles — kolom itu ada di all_events tapi tidak ikut saat kita filter dribbles tadi.
Cek dulu:

In [27]:
print('competition' in flair_dribbles.columns)
print('competition' in all_events.columns)

True
True


Keduanya ada! Berarti masalahnya bukan kolom hilang. Cek isi kolom competition di flair_dribbles:

In [29]:
print(flair_dribbles['competition'].value_counts(dropna=False))

competition
WC 2022              1076
Euro 2024             787
Copa America 2024     527
Name: count, dtype: int64



Kolom ada dan terisi dengan benar. Berarti masalahnya ada di all_matches — kemungkinan kolom competition di sana duplikat saat merge. Cek:

In [30]:
print(all_matches.columns.tolist())
print(all_matches['competition'].value_counts(dropna=False))

['match_id', 'competition']
competition
WC 2022              64
Euro 2024            51
Copa America 2024    32
Name: count, dtype: int64


Kedua dataframe punya kolom competition — itulah masalahnya. Saat merge, pandas membuat competition_x dan competition_y, sehingga kolom competition tidak ada lagi di hasil merge.
Solusinya sederhana — karena flair_dribbles sudah punya kolom competition, kita tidak perlu merge sama sekali:

In [31]:
stage_features = flair_dribbles.groupby(['player_id', 'competition']).agg(
    nutmegs_in_comp=('dribble_nutmeg', 'sum'),
    attempts_in_comp=('dribble_outcome', 'count')
).reset_index()

stage_pivot = stage_features.pivot_table(
    index='player_id',
    columns='competition',
    values='nutmegs_in_comp',
    fill_value=0
).reset_index()

stage_pivot.columns.name = None
stage_pivot = stage_pivot.rename(columns={
    'WC 2022': 'nutmegs_wc2022',
    'Euro 2024': 'nutmegs_euro2024',
    'Copa America 2024': 'nutmegs_copa2024'
})

print(stage_pivot.head())
print(stage_pivot.shape)

   player_id nutmegs_copa2024 nutmegs_euro2024 nutmegs_wc2022
0     2941.0                0                0            2.0
1     2972.0                0              0.0            0.0
2     2988.0                0              2.0            0.0
3     2995.0              4.0                0            3.0
4     3009.0                0              0.0            5.0
(403, 4)


perfcto, Sekarang gabungkan ke master_features:

In [32]:
master_features = master_features.merge(stage_pivot, on='player_id', how='left')

# Fill NaN dengan 0 untuk kolom nutmeg per kompetisi
master_features[['nutmegs_wc2022', 'nutmegs_euro2024', 'nutmegs_copa2024']] = \
    master_features[['nutmegs_wc2022', 'nutmegs_euro2024', 'nutmegs_copa2024']].fillna(0)

print(master_features.shape)
print(master_features.columns.tolist())

(153, 16)
['player_id', 'player', 'team', 'total_dribbles', 'successful_dribbles', 'total_nutmegs', 'total_overrun', 'dribble_success_rate', 'nutmeg_rate', 'matches_played', 'dribbles_per_match', 'total_matches', 'form_nutmeg_rate', 'nutmegs_copa2024', 'nutmegs_euro2024', 'nutmegs_wc2022']


153 pemain, 16 fitur — Feature Engineering Level 2 selesai!

In [33]:
master_features.to_csv('data/processed/master_features.csv', index=False)
print("Tersimpan!")
print(master_features.sort_values('total_nutmegs', ascending=False).head(10))

Tersimpan!
     player_id                           player       team  total_dribbles  \
3       2995.0  Ángel Fabián Di María Hernández  Argentina              47   
93     25104.0            Rodrygo Silva de Goes     Brazil              23   
138    68574.0        Nicholas Williams Arthuer      Spain              45   
149   316046.0      Lamine Yamal Nasraoui Ebana      Spain              27   
4       3009.0             Kylian Mbappé Lottin     France              77   
125    39565.0                    Jamal Musiala    Germany              42   
71     16528.0                 Petar Stojanović   Slovenia              11   
46      7156.0                  Federico Chiesa      Italy              13   
35      5503.0   Lionel Andrés Messi Cuccittini  Argentina              55   
72     16532.0             Daniel Olmo Carvajal      Spain              29   

     successful_dribbles  total_nutmegs  total_overrun  dribble_success_rate  \
3                     26              7           

In [34]:
print("=== MASTER FEATURES SUMMARY ===")
print(f"Shape: {master_features.shape}")
print(f"\nFitur yang tersedia:")
for col in master_features.columns:
    print(f"  - {col}")
print(f"\nMissing values:\n{master_features.isnull().sum()[master_features.isnull().sum() > 0]}")

=== MASTER FEATURES SUMMARY ===
Shape: (153, 16)

Fitur yang tersedia:
  - player_id
  - player
  - team
  - total_dribbles
  - successful_dribbles
  - total_nutmegs
  - total_overrun
  - dribble_success_rate
  - nutmeg_rate
  - matches_played
  - dribbles_per_match
  - total_matches
  - form_nutmeg_rate
  - nutmegs_copa2024
  - nutmegs_euro2024
  - nutmegs_wc2022

Missing values:
Series([], dtype: int64)
